In [ ]:
import numpy as np
import nibabel as nib
from pathlib import Path
from nilearn.image import resample_to_img
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns

# 設定路徑與目標 (請確保 face 與 bunkbed 的字串與你的 CSV 內完全相同)
base_dir = Path(r"D:\NTU_PSY\Documents\Patric_Asen_BrainHackProject")
sub_id = "sub-01"
targets = ["face", "bunkbed"]

# ==========================================
# 1. 讀取與篩選 Trial (使用你原有的 function)
# ==========================================
# 假設 load_and_filter_conditions 與 extract_fmri_volumes_rowwise 已在前面的 cell 定義
df_sorted = load_and_filter_conditions(base_dir, sub_id, targets, verbose=True)

# 提取這兩個 concept 所有的 fMRI beta volumes 
# shape 將會是 (x, y, z, n_trials)
volumes = extract_fmri_volumes_rowwise(df_sorted, base_dir, sub_id, verbose=True)

# ==========================================
# 2. 處理 HCP Atlas (Resampling)
# ==========================================
# 抓取第一張影像作為對齊參考
ref_img = nib.load(base_dir / f"subject_fMRI_nii/{sub_id}/ses-things01/{sub_id}_ses-things01_run-01_betas.nii")
atlas_img = nib.load(base_dir / "HCP_atlas/MNI_Glasser_HCP_v1.0.nii.gz")

# 將 Atlas 降採樣以符合 fMRI 的解析度
atlas_rs = resample_to_img(atlas_img, ref_img, interpolation="nearest")
atlas_data = atlas_rs.get_fdata()

# ==========================================
# 3. 建立 FFA 與 PPA 的 Mask
# ==========================================
ffa_ids = [18, 1018]
ppa_ids = [126, 127, 155, 1126, 1127, 1155]

mask_ffa = np.isin(atlas_data, ffa_ids)
mask_ppa = np.isin(atlas_data, ppa_ids)

# ==========================================
# 4. 提取各 ROI 的平均活化值並分類
# ==========================================
trial_concepts = df_sorted["concept"].values

ffa_face_acts, ffa_bunkbed_acts = [], []
ppa_face_acts, ppa_bunkbed_acts = [], []

for i, concept in enumerate(trial_concepts):
    vol = volumes[..., i] # 取出單一 trial 的大腦影像
    
    # 計算 ROI 內所有體素 (voxel) 的平均 beta 值
    ffa_mean = np.mean(vol[mask_ffa])
    ppa_mean = np.mean(vol[mask_ppa])
    
    if concept == "face":
        ffa_face_acts.append(ffa_mean)
        ppa_face_acts.append(ppa_mean)
    elif concept == "bunkbed":
        ffa_bunkbed_acts.append(ffa_mean)
        ppa_bunkbed_acts.append(ppa_mean)

# ==========================================
# 5. 進行統計檢定 (T-test) 與視覺化
# ==========================================
# 檢定 FFA 中 Face 是否與 Bunkbed 有顯著差異
t_ffa, p_ffa = ttest_ind(ffa_face_acts, ffa_bunkbed_acts)

# 檢定 PPA 中 Face 是否與 Bunkbed 有顯著差異
t_ppa, p_ppa = ttest_ind(ppa_face_acts, ppa_bunkbed_acts)

print(f"\n[FFA ROI 分析]")
print(f"Face 平均活化: {np.mean(ffa_face_acts):.4f} | Bunkbed 平均活化: {np.mean(ffa_bunkbed_acts):.4f}")
print(f"T-test: t = {t_ffa:.4f}, p = {p_ffa:.4e}")

print(f"\n[PPA ROI 分析]")
print(f"Face 平均活化: {np.mean(ppa_face_acts):.4f} | Bunkbed 平均活化: {np.mean(ppa_bunkbed_acts):.4f}")
print(f"T-test: t = {t_ppa:.4f}, p = {p_ppa:.4e}")

# (Optional) 畫個簡單的長條圖來確認雙重解離 (Double Dissociation) 的趨勢
data = [
    np.mean(ffa_face_acts), np.mean(ffa_bunkbed_acts),
    np.mean(ppa_face_acts), np.mean(ppa_bunkbed_acts)
]
errors = [
    np.std(ffa_face_acts)/np.sqrt(len(ffa_face_acts)), np.std(ffa_bunkbed_acts)/np.sqrt(len(ffa_bunkbed_acts)),
    np.std(ppa_face_acts)/np.sqrt(len(ppa_face_acts)), np.std(ppa_bunkbed_acts)/np.sqrt(len(ppa_bunkbed_acts))
]

fig, ax = plt.subplots(figsize=(6, 4))
bar_width = 0.35
x = np.arange(2)

ax.bar(x - bar_width/2, [data[0], data[2]], bar_width, yerr=[errors[0], errors[2]], label='Face', capsize=5)
ax.bar(x + bar_width/2, [data[1], data[3]], bar_width, yerr=[errors[1], errors[3]], label='Bunkbed', capsize=5)

ax.set_ylabel('Mean Beta Value')
ax.set_title('ROI Activation: Face vs Bunkbed')
ax.set_xticks(x)
ax.set_xticklabels(['FFA', 'PPA'])
ax.legend()
plt.tight_layout()
plt.show()
